# METRIC-RES-001 — checkpoint-only identity-resolution audit (Colab T4)

**What this decides.** FORMATION-MUX-001 S5 ran 24 arms for 8.14 GPU-hours and returned `INCONCLUSIVE_AT_ZERO_BASELINE` because the identity/copy family scored 0.0–0.008 in *every* arm. The returned-bundle audit prescribed exactly one next action: re-audit the **preserved checkpoints** at token resolution to learn whether identity is *forming-but-unmeasured* or *genuinely absent*.

**This trains nothing.** No optimizer, no backward, no parameter mutation. It loads the S5 weights read-only and re-scores them under a higher-resolution instrument (per-token accuracy, gold rank/margin, longest-common-prefix).

**Before you run**
1. Runtime → Change runtime type → **T4 GPU**.
2. **Keep this browser tab open for the whole run.** Colab 2026 Pro+ runtimes are documented to survive *longer* while the tab is open; background runs terminate at 3–10h with no error log.
3. Put the S5 checkpoints in Drive at `MyDrive/CYMEK/FORMATION_MUX_001/CS-MECH-002/<ARM>/S<1-4>/resume.pt` (16 files) plus `MyDrive/CYMEK/FORMATION_MUX_001/PUBLIC_SURFACE_MANIFEST.json`. These are **not** in the S5 result ZIP — they are still in the original Kaggle Output tree.
4. Expected wall time: **20–40 minutes**. Short by design, so the 12h Colab cap is not a risk.

**Cell 3 runs on CPU and is free.** Cell 2 needs the GPU.

In [ ]:
# CELL 1 — fetch pinned code, verify identity, no torch in this kernel
import os
# Architecture: allocator is set before any GPU process starts, and this
# kernel never imports torch, so it holds no CUDA context on the T4.
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ['PYTHONUNBUFFERED'] = '1'
import json, shutil, subprocess, sys
from pathlib import Path

REMOTE = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
BRANCH = 'cymek-next-core-architecture'
EXECUTION_COMMIT = 'REPLACE_AFTER_COMMIT'
OPERATOR_BLOB = 'REPLACE_AFTER_COMMIT'
PREREG_BLOB = 'REPLACE_AFTER_COMMIT'
REPO = Path('/content/An-Ra-the-new-AGI-metric-res')
OP_COPY = Path('/content/metric_resolution_audit_v1.py')

if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REMOTE,str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','-q',EXECUTION_COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==EXECUTION_COMMIT
def blob(rel):
    return subprocess.check_output(['git','-C',str(REPO),'hash-object',rel],text=True).strip()
assert blob('tools/metric_resolution_audit_v1.py')==OPERATOR_BLOB, (blob('tools/metric_resolution_audit_v1.py'),OPERATOR_BLOB)
assert blob('docs/cymek/experiments/METRIC-RES-001/PREREGISTRATION.json')==PREREG_BLOB
shutil.copy2(REPO/'tools/metric_resolution_audit_v1.py', OP_COPY)
try:
    __import__('tokenizers')
except ImportError:
    subprocess.run([sys.executable,'-m','pip','install','-q','tokenizers'],check=True)
# Hardware gate: verify T4 via nvidia-smi (no torch import, no CUDA context).
smi = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader,nounits'],capture_output=True,text=True)
gpus = [l.strip() for l in smi.stdout.strip().splitlines() if l.strip()] if smi.returncode==0 else []
print('GPUs visible:', gpus)
if len(gpus)!=1 or 'T4' not in gpus[0]:
    raise RuntimeError('AUDIT BLOCKED: select Runtime -> Change runtime type -> T4 GPU (observed: '+str(gpus)+')')
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/CYMEK/FORMATION_MUX_001')
LOCAL_OUT = Path('/content/METRIC_RES_001')
print('READY | checkpoints root:', DRIVE_ROOT)
print('local working dir:', LOCAL_OUT, '(written first, synced to Drive with retry)')

In [ ]:
# CELL 2 — GPU stage: audit all 16 checkpoints (4 arms x 4 seeds)
import subprocess, sys, time
from pathlib import Path
LOCAL_OUT = Path('/content/METRIC_RES_001')
LOCAL_OUT.mkdir(parents=True, exist_ok=True)
cmd = [sys.executable,'-u',str(OP_COPY),
       '--repo',str(REPO),'--out',str(LOCAL_OUT),
       '--checkpoints',str(DRIVE_ROOT),'--device','cuda']
print('Starting METRIC-RES-001 GPU stage. Expect 20-40 min. Keep this tab open.',flush=True)
t0 = time.monotonic()
rc = subprocess.run(cmd, cwd=REPO).returncode
print('GPU STAGE RETURN CODE:', rc, '| elapsed min:', round((time.monotonic()-t0)/60,1), flush=True)
if rc != 0:
    fail = LOCAL_OUT/'METRIC_RES_FAILURE.json'
    if fail.exists(): print('FAILURE:', fail.read_text())
    raise SystemExit('GPU stage failed closed. Local receipts are preserved in /content/METRIC_RES_001.')
receipts = sorted(LOCAL_OUT.glob('ARM_*.json'))
print('arm receipts:', len(receipts), '(expect 16)')

In [ ]:
# CELL 3 — CPU stage (FREE, no GPU needed): decide + readjudicate + package
# You may switch Runtime -> Accelerator -> None before this cell.
# Imports no torch, so it cannot consume GPU quota.
import json, subprocess, sys
from pathlib import Path
LOCAL_OUT = Path('/content/METRIC_RES_001')
DRIVE_OUT = Path('/content/drive/MyDrive/CYMEK/METRIC_RES_001')
cmd = [sys.executable,'-u',str(OP_COPY),
       '--repo',str(REPO),'--out',str(LOCAL_OUT),'--checkpoints',str(DRIVE_ROOT),
       '--aggregate-only','--sync-target',str(DRIVE_OUT)]
rc = subprocess.run(cmd, cwd=REPO).returncode
if rc != 0:
    raise SystemExit('Aggregate stage failed closed. Do not interpret partial results.')
print()
d = json.loads((LOCAL_OUT/'DECISION.json').read_text())
print('DECISION:', d['decision']['decision'])
print('MEASURED:', json.dumps(d['decision']['measured'], indent=2))
print('CLAIM CEILING:', d['claim_ceiling'])

In [ ]:
# CELL 4 — download the small results bundle for the record
from google.colab import files
bundle = Path('/content/METRIC_RES_001_RESULTS.zip')
if not bundle.exists():
    raise RuntimeError('No results bundle. Re-run Cell 3.')
print('Bundle bytes:', bundle.stat().st_size)
files.download(str(bundle))